# Feature Engineering

This notebook takes the cleaned sequence data and builds the feature matrix used for modeling.

We will:
1. Classify hosts into broad animal groups (bat, primate, rodent, etc.)
2. Encode molecule type as a number
3. Merge in phylogenetic risk scores from the genomics analysis
4. Save the final feature matrix to `data/processed/features.csv`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load cleaned data

In [2]:
df = pd.read_csv('../../data/processed/sequences_cleaned.csv', low_memory=False)
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
print(df.columns.tolist())
df.head()

Rows: 3,256,603  |  Columns: 16
['Accession', 'Organism_Name', 'Species', 'Isolate', 'Length', 'Nuc_Completeness', 'Geo_Location', 'Host', 'Tissue_Specimen_Source', 'Submitters', 'Collection_Date', 'Release_Date', 'Molecule_type', 'Year', 'Month', 'Continent']


,Accession,Organism_Name,Species,Isolate,Length,Nuc_Completeness,Geo_Location,Host,Tissue_Specimen_Source,Submitters,Collection_Date,Release_Date,Molecule_type,Year,Month,Continent
0,NC_138438,Red panda feces-associated circular DNA virus 12,Alohovirus ailgensis,Rpf011unssDNA01-5,2820,complete,China: Sichuan Province,Ailurus fulgens,feces,"Zhao,M., Yue,C., Yang,Z., Li,Y., Zhang,D., Zha...",2020-05-01,2026-04-07,ssDNA,2020,5,Asia
1,NC_138461,Red panda feces-associated circular DNA virus 11,Protegovirus ailuris,AliP02cress04-2015,2497,complete,China: Sichuan Province,Ailurus fulgens,feces,"Zhao,M., Yue,C., Yang,Z., Li,Y., Zhang,D., Zha...",2015-03-01,2026-04-07,ssDNA,2015,3,Asia
2,NC_097195,Pacific flying fox faeces associated circular ...,Pekavirus fuais,Tbat_38855,2214,complete,Tonga,Pteropus tonganus,feces,"Male,M.F., Kraberger,S., Stainton,D., Kami,V.,...",2014-06-01,2026-04-01,ssDNA,2014,6,Oceania
3,NC_099061,Bat circovirus POA/2012/V,Vegetinivirus molmolis,cluster V,1728,complete,Brazil,Chiroptera,feces,"Lima,F.E., Cibulski,S.P., Dos Santos,H.F., Tei...",2012-02-13,2026-04-01,ssDNA,2012,2,South America
4,NC_116632,Delphin virus 1,Baranavirus uduis,3_2016_1939,1939,complete,Saint Vincent and the Grenadines,Orcinus orca,muscle,"Smith,K., Fielding,R., Schiavone,K., Hall,K., ...",2015-08-26,2026-04-01,ssDNA,2015,8,North America


## Classify hosts into animal groups

The `Host` column has scientific names like *Rhinolophus sinicus*. We map these to broad groups (bat, primate, rodent, etc.) so we can use them in the model and visualizations.

Anything not matched gets labeled `other`.

In [3]:
# keywords to match against the Host column (lowercase)
HOST_GROUPS = {
    'bat':      ['rhinolophus', 'pteropus', 'myotis', 'eptesicus', 'tadarida', 'rousettus',
                 'hipposideros', 'pipistrellus', 'molossus', 'miniopterus', 'chiroptera',
                 'carollia', 'artibeus', 'desmodus', 'chaerephon', 'mops', 'nyctalus',
                 'murina', 'tylonycteris', 'bat'],
    'primate':  ['homo', 'pan', 'gorilla', 'pongo', 'macaca', 'papio', 'cercopithecus',
                 'mandrillus', 'callithrix', 'saimiri', 'aotus', 'primate', 'monkey', 'ape'],
    'rodent':   ['mus', 'rattus', 'peromyscus', 'cricetulus', 'microtus', 'sigmodon',
                 'myodes', 'apodemus', 'marmota', 'cavia', 'hydrochoerus', 'rodent', 'mouse', 'rat'],
    'bird':     ['gallus', 'anas', 'anser', 'columba', 'corvus', 'aves', 'duck', 'goose',
                 'chicken', 'bird', 'avian'],
    'pig':      ['sus scrofa', 'sus', 'swine', 'pig', 'porcine'],
    'cattle':   ['bos ', 'bos_', 'bovine', 'cattle', 'cow'],
    'carnivore':['felis', 'canis', 'vulpes', 'mustela', 'panthera', 'cat', 'dog', 'fox',
                 'mink', 'ferret'],
    'pangolin': ['manis', 'smutsia', 'phataginus', 'pangolin'],
    'human':    ['homo sapiens', 'human', 'humans'],
}

def classify_host(host):
    if pd.isna(host):
        return 'unknown'
    host_lower = str(host).lower()
    for group, keywords in HOST_GROUPS.items():
        if any(kw in host_lower for kw in keywords):
            return group
    return 'other'

df['host_group'] = df['Host'].apply(classify_host)

print(df['host_group'].value_counts())

host_group
primate      3222951
pig            13747
other           8699
cattle          3516
carnivore       3376
rodent          2362
bat             1925
pangolin          25
bird               2
Name: count, dtype: int64


## Encode molecule type

`Molecule_type` tells us the type of genetic material the virus uses. We convert it to a number so it can be used in models.

In [5]:
print('Molecule types found:')
print(df['Molecule_type'].value_counts())

molecule_map = {
    'ssDNA':  1,
    'dsDNA':  2,
    'ssRNA':  3,
    'dsRNA':  4,
    'RNA':    3,   # treat unspecified RNA as ssRNA
    'DNA':    2,   # treat unspecified DNA as dsDNA
}
df['molecule_type_code'] = df['Molecule_type'].map(molecule_map).fillna(0).astype(int)
print('\nEncoded:')
print(df['molecule_type_code'].value_counts())

Molecule types found:
Molecule_type
ssRNA(+)      3178602
ssRNA(-)        25639
dsDNA           11976
ssDNA(+/-)       7689
dsDNA-RT         7128
ssRNA-RT         7106
dsRNA            7039
ssRNA(+/-)       5440
ssDNA(-)         4395
ssDNA            1264
unknown           283
RNA                34
DNA                 7
ssRNA               1
Name: count, dtype: int64

Encoded:
molecule_type_code
0    3236282
2      11983
4       7039
1       1264
3         35
Name: count, dtype: int64


## Merge in phylogenetic risk scores

We use the same scoring formula from `genomics.ipynb`:

> *risk score = e^(−divergence_mya / 50)*

Rather than calling the NCBI API again for every row, we apply scores at the **host group level** using known divergence times. Individual species scores can be substituted here if available from the genomics notebook output.

In [6]:
# Divergence times (MYA) from humans — TimeTree consensus
GROUP_DIVERGENCE_MYA = {
    'human':    0.0,
    'primate':  29.0,   # Old World monkeys; closer groups (Pan=6.9) will raise this
    'pangolin': 80.0,
    'rodent':   87.0,
    'carnivore': 96.0,
    'pig':      95.0,
    'cattle':   95.0,
    'bat':      95.0,
    'bird':     312.0,
    'other':    150.0,  # conservative middle estimate
    'unknown':  150.0,
}

df['divergence_mya']    = df['host_group'].map(GROUP_DIVERGENCE_MYA)
df['phylo_risk_score']  = np.exp(-df['divergence_mya'] / 50)

print(df.groupby('host_group')[['divergence_mya', 'phylo_risk_score']].first().sort_values('phylo_risk_score', ascending=False))

            divergence_mya  phylo_risk_score
host_group                                  
primate               29.0          0.559898
pangolin              80.0          0.201897
rodent                87.0          0.175520
bat                   95.0          0.149569
cattle                95.0          0.149569
pig                   95.0          0.149569
carnivore             96.0          0.146607
other                150.0          0.049787
bird                 312.0          0.001950


## Encode geographic region

The `Continent` column was already added during cleaning. We encode it as a number here.

In [7]:
print('Continents found:')
print(df['Continent'].value_counts())

continent_map = {
    'Asia':          1,
    'Europe':        2,
    'North America': 3,
    'South America': 4,
    'Africa':        5,
    'Oceania':       6,
    'Antarctica':    7,
}
df['continent_code'] = df['Continent'].map(continent_map).fillna(0).astype(int)
print('\nEncoded:')
print(df['continent_code'].value_counts())

Continents found:
Continent
Europe           1633505
North America    1477254
Asia               80749
South America      29320
Africa             21098
Oceania            13866
Antarctica           475
Name: count, dtype: int64

Encoded:
continent_code
2    1633505
3    1477254
1      80749
4      29320
5      21098
6      13866
7        475
0        336
Name: count, dtype: int64


## Encode nucleotide completeness

Whether the genome sequence is complete or partial — useful as a data quality flag.

In [8]:
print(df['Nuc_Completeness'].value_counts())

df['is_complete'] = (df['Nuc_Completeness'] == 'complete').astype(int)
print(f'\nComplete sequences: {df["is_complete"].sum():,} / {len(df):,}')

Nuc_Completeness
complete    3256603
Name: count, dtype: int64

Complete sequences: 3,256,603 / 3,256,603


## Save feature matrix

Select the final columns and save to `data/processed/features.csv`.

In [9]:
features = df[[
    'Accession',
    'Host',
    'host_group',
    'divergence_mya',
    'phylo_risk_score',
    'Molecule_type',
    'molecule_type_code',
    'Continent',
    'continent_code',
    'Geo_Location',
    'Length',
    'is_complete',
    'Year',
    'Month',
]].copy()

features.to_csv('../../data/processed/features.csv', index=False)
print(f'Saved {len(features):,} rows to data/processed/features.csv')
print(f'\nColumn summary:')
features.describe(include='all').T[['count', 'unique', 'top', 'mean']].to_string()

Saved 3,256,603 rows to data/processed/features.csv

Column summary:


'                        count   unique                     top          mean\nAccession             3256603  3256603               NC_138438           NaN\nHost                  3256603      754            Homo sapiens           NaN\nhost_group            3256603        9                 primate           NaN\ndivergence_mya      3256603.0      NaN                     NaN     29.824177\nphylo_risk_score    3256603.0      NaN                     NaN      0.555408\nMolecule_type         3256603       14                ssRNA(+)           NaN\nmolecule_type_code  3256603.0      NaN                     NaN      0.016425\nContinent             3256267        7                  Europe           NaN\ncontinent_code      3256603.0      NaN                     NaN      2.483819\nGeo_Location          3256267     5962  United Kingdom:England           NaN\nLength              3256603.0      NaN                     NaN  29466.621555\nis_complete         3256603.0      NaN                     NaN 

In [10]:
features.head(10)

,Accession,Host,host_group,divergence_mya,phylo_risk_score,Molecule_type,molecule_type_code,Continent,continent_code,Geo_Location,Length,is_complete,Year,Month
0,NC_138438,Ailurus fulgens,other,150.0,0.049787,ssDNA,1,Asia,1,China: Sichuan Province,2820,1,2020,5
1,NC_138461,Ailurus fulgens,other,150.0,0.049787,ssDNA,1,Asia,1,China: Sichuan Province,2497,1,2015,3
2,NC_097195,Pteropus tonganus,bat,95.0,0.149569,ssDNA,1,Oceania,6,Tonga,2214,1,2014,6
3,NC_099061,Chiroptera,bat,95.0,0.149569,ssDNA,1,South America,4,Brazil,1728,1,2012,2
4,NC_116632,Orcinus orca,other,150.0,0.049787,ssDNA,1,North America,3,Saint Vincent and the Grenadines,1939,1,2015,8
5,NC_099376,Miniopterus fuliginosus,bat,95.0,0.149569,ssDNA,1,Asia,1,China,1643,1,2013,4
6,NC_099378,Plecotus auritus,other,150.0,0.049787,ssDNA,1,Asia,1,China,1881,1,2013,6
7,NC_129432,Miniopterus fuliginosus,bat,95.0,0.149569,ssDNA,1,Asia,1,China,1728,1,2012,12
8,NC_138392,Myotis emarginatus,bat,95.0,0.149569,ssDNA,1,Europe,2,Hungary,1638,1,2013,4
9,NC_138394,Chiroptera,bat,95.0,0.149569,ssDNA,1,North America,3,USA,1696,1,2008,8
